<a id="0"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#255E58;overflow:hidden">Análise Avançada de Dados para IoT  📊📈</div>

<a id="1"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#254E58;overflow:hidden">Módulo 11 → Análise de Dados em Streaming
</div>

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Importando
Bibliotecas</b></span>

In [ ]:
# BIBLIOTECAS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


# CONFIGURAÇÕES DE ESTÉTICA👇🏽

# Ajustes visuais do Jupyter
from IPython.core.display import HTML
display(HTML('<style>.container { width:80% !important; }</style>'))
# pd.set_option('display.max_columns', 60)

# Configurando pandas para mostrar todas as linhas e colunas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) 

# Configurando pandas para não mostrar notação científica para números
pd.set_option('display.float_format', lambda x: '%.2f' % x)
# Serve para Manter os gráficos nesse mesmo Notebook
%matplotlib inline
# Ignorando alertas
import warnings
warnings.filterwarnings("ignore")

# Reprodutibilidade
np.random.seed(42)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 1

In [ ]:
# 1. DADOS
df = pd.DataFrame({
    "tempo": pd.date_range("2026-09-29 08:00", periods=60, freq="1min"),
    "temperatura": 30 + np.random.normal(0, 0.8, 60)
})

df.loc[35:39, "temperatura"] += [2, 4, 6, 7, 5]

limite = 34
df["alerta_streaming"] = df["temperatura"] >= limite

# 2. BATCH
batch = (
    df.set_index("tempo")["temperatura"]
    .resample("10min")
    .mean()
    .reset_index()
)

batch["alerta_batch"] = batch["temperatura"] >= limite

# 3. STREAMING
alertas = df[df["alerta_streaming"]]

plt.figure(figsize=(10, 4))
plt.plot(df["tempo"], df["temperatura"], marker="o", markersize=3, label="Leituras em tempo real")
plt.axhline(limite, color="red", linestyle="--", label="Limite = 34 °C")
plt.scatter(alertas["tempo"], alertas["temperatura"], color="red", s=80, label="Alerta imediato")
plt.title("Streaming: análise leitura a leitura")
plt.xlabel("Tempo")
plt.ylabel("Temperatura (°C)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 4. BATCH
plt.figure(figsize=(10, 4))
plt.bar(batch["tempo"].dt.strftime("%H:%M"), batch["temperatura"], color="orange")
plt.axhline(limite, color="red", linestyle="--", label="Limite = 34 °C")
plt.title("Batch: média processada a cada 10 minutos")
plt.xlabel("Janela de processamento")
plt.ylabel("Temperatura média (°C)")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

# 5. RESULTADOS
print("Alertas no streaming:", df["alerta_streaming"].sum())
print("Lotes com alerta no batch:", batch["alerta_batch"].sum())

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 4

### **Gráfico**

In [ ]:
def gerar_dados():
    tempo = np.linspace(0, 10, 200)

    tempo_real = 1 + 9 / (1 + np.exp(-(tempo - 4)))
    com_atraso = 1 + 8 / (1 + np.exp(-(tempo - 5)))
    referencia = 1 + 0.03 * tempo

    return tempo, tempo_real, com_atraso, referencia


def plotar(tempo, tempo_real, com_atraso, referencia):
    plt.figure(figsize=(10, 6))

    plt.plot(tempo, tempo_real, color="blue", linewidth=3, label="Resposta em Tempo Real")
    plt.plot(tempo, com_atraso, color="orange", linestyle="--", linewidth=3, label="Resposta com Atraso")
    plt.plot(tempo, referencia, color="green", linewidth=3, label="Sinal de referência")

    plt.axhline(10, color="dodgerblue", linestyle="--", linewidth=1.5)
    plt.text(0.05, 9.85, "Limite", fontsize=12)

    plt.title("Processamento em Tempo Real em IoT", fontsize=16)
    plt.xlabel("Tempo")
    plt.ylabel("Resposta")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


tempo, tempo_real, com_atraso, referencia = gerar_dados()
plotar(tempo, tempo_real, com_atraso, referencia)

### **Exemplo de Aplicação**

In [ ]:
def gerar_dados():
    tempo = np.arange(0, 121, 1)
    limite = 250

    lote = np.piecewise(
        tempo,
        [tempo <= 40, (tempo > 40) & (tempo <= 60), tempo > 60],
        [
            lambda t: 180 + 1.0*t,
            lambda t: 220 + 3.0*(t - 40),
            lambda t: 280 + 0.5*(t - 60)
        ]
    )

    critico = tempo[np.where(lote > limite)[0][0]]

    tempo_real = lote.copy()
    tempo_real[tempo >= critico] = (
        lote[np.where(tempo == critico)[0][0]]
        - 2.5*(tempo[tempo >= critico] - critico)
    )
    tempo_real = np.maximum(tempo_real, 180)

    instantes = np.array([0, 60, 120])
    temp_lote = lote[instantes]

    deteccao_lote = 60
    deteccao_real = critico
    atraso_lote = deteccao_lote - critico

    aquecedor_lote = np.ones_like(tempo)
    aquecedor_real = np.ones_like(tempo)
    aquecedor_real[tempo >= critico] = 0

    return (
        tempo, limite, lote, tempo_real, critico,
        instantes, temp_lote, deteccao_lote,
        deteccao_real, atraso_lote,
        aquecedor_lote, aquecedor_real
    )


def finalizar(titulo, ylabel):
    plt.title(titulo)
    plt.xlabel("Tempo (minutos)")
    plt.ylabel(ylabel)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


(
    tempo, limite, lote, tempo_real, critico,
    instantes, temp_lote, deteccao_lote,
    deteccao_real, atraso_lote,
    aquecedor_lote, aquecedor_real
) = gerar_dados()


# 1. TEMPERATURA
plt.figure(figsize=(10, 4))
plt.plot(tempo, lote, linewidth=2, label="Processamento em lote")
plt.plot(tempo, tempo_real, linewidth=2, label="Processamento em tempo real")
plt.axhline(limite, linestyle="--", linewidth=2, label="Limite seguro (250°C)")
plt.axvline(critico, linestyle=":", linewidth=2,
            label=f"Ultrapassou o limite ({critico} min)")
finalizar("Temperatura do forno ao longo do tempo", "Temperatura (°C)")


# 2. MOMENTO DA DETECÇÃO
plt.figure(figsize=(10, 4))
plt.plot(tempo, lote, linewidth=2, label="Temperatura do forno")
plt.axhline(limite, linestyle="--", linewidth=2, label="Limite seguro")
plt.scatter(instantes, temp_lote, s=80, label="Leituras em lote (0, 60, 120 min)")
plt.axvline(deteccao_real, linestyle=":", linewidth=2, label="Detecção em tempo real")
plt.axvline(deteccao_lote, linestyle=":", linewidth=2, label="Detecção em lote")
finalizar("Comparação da detecção do problema", "Temperatura (°C)")


# 3. ESTADO DO AQUECEDOR
plt.figure(figsize=(10, 4))
plt.step(tempo, aquecedor_lote, where="post", linewidth=2,
         label="Aquecedor com processamento em lote")
plt.step(tempo, aquecedor_real, where="post", linewidth=2,
         label="Aquecedor com processamento em tempo real")
plt.yticks([0, 1], ["Desligado", "Ligado"])
finalizar("Estado do aquecedor", "Estado (1 = ligado | 0 = desligado)")


# RESUMO
print(f"Temperatura ultrapassou 250°C no minuto: {critico}")
print(f"Detecção em tempo real: minuto {deteccao_real}")
print(f"Detecção em lote: minuto {deteccao_lote}")
print(f"Atraso do lote: {atraso_lote} minutos")

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 5

In [ ]:
def gerar_dados():
    np.random.seed(42)

    df = pd.DataFrame({
        "tempo": pd.date_range("2026-10-02 08:00", periods=40, freq="1min"),
        "temperatura": 5 + np.random.normal(0, 0.35, 40)
    })

    df.loc[20:25, "temperatura"] += [1.0, 1.8, 2.6, 3.0, 2.4, 1.6]
    return df


def detectar_alertas(df, limite=7.0):
    df["acima_limite"] = df["temperatura"] > limite
    df["alerta"] = df["acima_limite"].rolling(3).sum().eq(3)
    return df


df = detectar_alertas(gerar_dados())
limite = 7.0

# 1. DETECÇÃO EM FLUXO
eventos = df[df["acima_limite"]]

plt.figure(figsize=(10, 4))
plt.plot(df["tempo"], df["temperatura"], color="blue", marker="o",
         markersize=4, label="Temperatura")
plt.axhline(limite, color="orange", linestyle="--",
            linewidth=2, label="Limite = 7 °C")
plt.scatter(eventos["tempo"], eventos["temperatura"],
            color="red", s=80, label="Evento detectado")
plt.title("Detecção em Fluxo")
plt.xlabel("Tempo")
plt.ylabel("Temperatura (°C)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 2. ALERTA
plt.figure(figsize=(10, 4))
plt.step(df["tempo"], df["alerta"].astype(int),
         where="post", color="red", linewidth=2)
plt.yticks([0, 1], ["Normal", "ALERTA"])
plt.title("Alerta após 3 Leituras Consecutivas")
plt.xlabel("Tempo")
plt.ylabel("Estado")
plt.grid(alpha=0.3)
plt.show()

# 3. RESULTADOS
print("Leituras acima do limite:", df["acima_limite"].sum())
print("Momentos com alerta:", df["alerta"].sum())

**Conclusão**

* A detecção em fluxo encontra eventos conforme os dados chegam; o alerta é disparado quando esses eventos atendem uma regra definida.

## **Outro**

In [ ]:
def gerar_dados():
    tempo = np.arange(150)
    temperatura = np.zeros(150) - 5

    for centro, altura, largura in [(35, 20, 10), (75, 16, 9), (110, 13, 8)]:
        temperatura += altura * np.exp(-((tempo - centro) / largura) ** 2)

    return tempo, temperatura


tempo, temperatura = gerar_dados()

limiar = 7
pontos_x = [15, 75, 110]
pontos_y = temperatura[pontos_x]

plt.figure(figsize=(8, 4))
plt.plot(tempo, temperatura, color="yellowgreen", linewidth=2.5)
plt.axhline(limiar, color="tomato", linestyle="--", linewidth=2)
plt.scatter(pontos_x, pontos_y, color="tomato", s=80, zorder=5)

plt.title("Detecção de Alertas em Fluxo")
plt.xlabel("Tempo")
plt.ylabel("Temperatura")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

##### ALERTAS E DETECÇÃO EM FLUXO
Projetos de IoT – Conceitos, Código e Aplicação Prática

##### O QUE É ALERTA?
Notificação disparada quando um valor cruza um limite pré-definido.
Exemplo: temperatura > 40 °C aciona um e-mail.

##### O QUE É DETECÇÃO?
Identificação automática de padrões ou anomalias no fluxo de dados em tempo real.
Exemplo: algoritmo percebe comportamento anormal antes que o limite seja atingido.

##### APLICAÇÕES PRÁTICAS
Monitoramento industrial, saúde wearable, cidades inteligentes, agricultura de precisão e segurança predial.

##### SITUAÇÃO-PROBLEMA
Uma fábrica precisa monitorar a temperatura de um forno industrial 24 horas por dia. Se a temperatura ultrapassar 40 °C, um alerta deve ser enviado imediatamente para evitar danos ao equipamento.

##### PASSOS DA SOLUÇÃO
1. Sensor envia leitura a cada segundo.
2. Sistema compara a leitura com o limite de 40 °C.
3. Se leitura > 40 °C → dispara alerta por e-mail/SMS.
4. Registro do evento em log para auditoria.

##### **Conclusão**
* O gráfico mostra como um sistema IoT pode acompanhar continuamente um sinal, identificar mudanças no comportamento e comparar essas leituras com um limite para gerar alertas.

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 6

In [ ]:
def gerar_dados(n=60, limite=2.0):
    np.random.seed(42)

    df = pd.DataFrame({
        "mensagem": np.arange(1, n + 1),
        "latencia_s": np.random.uniform(0.2, 1.2, n),
        "recebida": True
    })

    df.loc[[12, 27, 41, 52], "latencia_s"] += [2.8, 3.5, 2.4, 4.0]
    df.loc[[18, 44, 57], "recebida"] = False
    df["atrasada"] = (df["latencia_s"] > limite) & df["recebida"]

    return df


df = gerar_dados()
limite = 2.0

taxa_entrega = df["recebida"].mean() * 100
taxa_no_prazo = (df["recebida"] & (df["latencia_s"] <= limite)).mean() * 100
atrasadas = df[df["atrasada"]]

# 1. LATÊNCIA
plt.figure(figsize=(10, 4))
plt.plot(df["mensagem"], df["latencia_s"], marker="o", color="blue", label="Latência")
plt.axhline(limite, color="orange", linestyle="--", label="Limite aceitável = 2 s")
plt.scatter(atrasadas["mensagem"], atrasadas["latencia_s"], color="red", s=90, label="Mensagem atrasada")
plt.title("Latência das Mensagens IoT")
plt.xlabel("Número da mensagem")
plt.ylabel("Latência (s)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 2. CONFIABILIDADE
plt.figure(figsize=(7, 4))
plt.bar(["Entregues", "Entregues no prazo"], [taxa_entrega, taxa_no_prazo])
plt.ylim(0, 100)
plt.title("Confiabilidade da Comunicação IoT")
plt.ylabel("Percentual (%)")
plt.grid(axis="y", alpha=0.3)
plt.show()

# 3. RESULTADOS
print("Mensagens enviadas:", len(df))
print("Mensagens entregues:", df["recebida"].sum())
print("Mensagens perdidas:", (~df["recebida"]).sum())
print("Mensagens atrasadas:", df["atrasada"].sum())
print(f"Taxa de entrega: {taxa_entrega:.1f}%")
print(f"Taxa no prazo: {taxa_no_prazo:.1f}%")

In [ ]:
categorias = [
    "Mensagens enviadas", "Mensagens entregues",
    "Mensagens perdidas", "Mensagens atrasadas",
    "Taxa de entrega (%)", "Taxa no prazo (%)"
]

valores = [60, 57, 3, 4, 95.0, 88.3]
cores = ["blue", "green", "red", "orange", "purple", "cyan"]


def plotar_resumo():
    plt.figure(figsize=(10, 5))

    barras = plt.bar(categorias, valores, color=cores)

    for barra, valor in zip(barras, valores):
        plt.text(
            barra.get_x() + barra.get_width() / 2,
            barra.get_height() + 1,
            f"{valor}",
            ha="center"
        )

    plt.title("Resumo da Comunicação IoT")
    plt.ylabel("Valor")
    plt.xticks(rotation=20)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


plotar_resumo()

**Conclusão**

* O sistema apresenta boa entrega geral, mas ainda possui atrasos e perdas que podem comprometer aplicações críticas.